In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import json
import math
from collections import Counter, defaultdict

SUMMARY_PATH = "/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/object_type_inspection_20260602_164308/_object_type_summary.json"

with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    summary = json.load(f)

In [ ]:
# Scene grouping

def scene_group(scene: str) -> str:
    number = int(scene.replace("FloorPlan", ""))

    if 1 <= number <= 30:
        return "kitchen"
    if 201 <= number <= 230:
        return "living_room"
    if 301 <= number <= 330:
        return "bedroom"
    if 401 <= number <= 430:
        return "bathroom"

    return "unknown"


successful_scenes = [
    r["scene"]
    for r in summary["scene_results"]
    if r["status"] == "success"
]

successful_scene_set = set(successful_scenes)

room_success_scenes = {
    "kitchen": [],
    "living_room": [],
    "bedroom": [],
    "bathroom": [],
}

for scene in successful_scenes:
    group = scene_group(scene)
    if group in room_success_scenes:
        room_success_scenes[group].append(scene)

room_success_counts = {
    room: len(scenes)
    for room, scenes in room_success_scenes.items()
}

num_success = summary["num_scenes_success"]

print("Successful scenes:", num_success)
print("Room success counts:", room_success_counts)
print("Failed scenes:", [x["scene"] for x in summary["failed_scenes"]])

Successful scenes: 115
Room success counts: {'kitchen': 29, 'living_room': 29, 'bedroom': 27, 'bathroom': 30}
Failed scenes: ['FloorPlan7', 'FloorPlan223', 'FloorPlan307', 'FloorPlan312', 'FloorPlan315']


In [ ]:
# Thresholds

STRONG_GLOBAL_COVERAGE = 0.20
STRONG_ROOM_COVERAGE = 0.50
MEDIUM_ROOM_COVERAGE = 0.33
WEAK_GLOBAL_COVERAGE = 0.10

strong_global_min_scene_count = math.ceil(STRONG_GLOBAL_COVERAGE * num_success)

print("Strong global min scene count:", strong_global_min_scene_count)


# Build quantitative objectType table

global_counts = summary["global_object_type_counts"]
presence = summary["object_type_scene_presence"]

rows = []

for object_type, instance_count in global_counts.items():
    scene_list = presence[object_type]["scenes"]
    scene_count = presence[object_type]["num_scenes"]

    group_counts = Counter(scene_group(scene) for scene in scene_list)

    room_coverage = {}
    for room, denom in room_success_counts.items():
        if denom == 0:
            room_coverage[room] = 0.0
        else:
            room_coverage[room] = group_counts[room] / denom

    global_coverage = scene_count / num_success

    max_room = max(room_coverage, key=room_coverage.get)
    max_room_coverage = room_coverage[max_room]

    strong_support = (
        scene_count >= strong_global_min_scene_count
        or max_room_coverage >= STRONG_ROOM_COVERAGE
    )

    medium_support = (
        not strong_support
        and max_room_coverage >= MEDIUM_ROOM_COVERAGE
    )

    weak_support = (
        not strong_support
        and not medium_support
    )

    if strong_support:
        support_level = "strong"
    elif medium_support:
        support_level = "medium"
    else:
        support_level = "weak"

    rows.append({
        "object_type": object_type,
        "instance_count": instance_count,
        "scene_count": scene_count,
        "global_coverage": global_coverage,
        "kitchen_scene_count": group_counts["kitchen"],
        "living_room_scene_count": group_counts["living_room"],
        "bedroom_scene_count": group_counts["bedroom"],
        "bathroom_scene_count": group_counts["bathroom"],
        "kitchen_coverage": room_coverage["kitchen"],
        "living_room_coverage": room_coverage["living_room"],
        "bedroom_coverage": room_coverage["bedroom"],
        "bathroom_coverage": room_coverage["bathroom"],
        "max_room": max_room,
        "max_room_coverage": max_room_coverage,
        "support_level": support_level,
    })

Strong global min scene count: 23


In [ ]:
# Metadata example statistics

metadata_stats = defaultdict(Counter)

for scene_result in summary["scene_results"]:
    if scene_result["status"] != "success":
        continue

    examples_by_type = scene_result.get("object_type_examples", {})

    for object_type, examples in examples_by_type.items():
        for ex in examples:
            for field in [
                "pickupable",
                "receptacle",
                "moveable",
                "openable",
                "visible",
            ]:
                metadata_stats[object_type][f"{field}={ex.get(field)}"] += 1


def metadata_majority(object_type, field):
    true_count = metadata_stats[object_type][f"{field}=True"]
    false_count = metadata_stats[object_type][f"{field}=False"]

    if true_count == 0 and false_count == 0:
        return None

    return true_count >= false_count


for row in rows:
    object_type = row["object_type"]
    row["pickupable_majority"] = metadata_majority(object_type, "pickupable")
    row["receptacle_majority"] = metadata_majority(object_type, "receptacle")
    row["moveable_majority"] = metadata_majority(object_type, "moveable")
    row["openable_majority"] = metadata_majority(object_type, "openable")

In [ ]:
# Objects excluded from all supplement sets because they are global background.
GLOBAL_EXCLUDED_TYPES = {
    "Floor",
    "Wall",
    "Ceiling",
}

# Ordinary relation participants.
# These should generally remain small/ordinary objects, not become structural anchors.
RELATION_PARTICIPANT_TYPES = {
    "Apple",
    "Bread",
    "ButterKnife",
    "Egg",
    "Fork",
    "Knife",
    "Lettuce",
    "PepperShaker",
    "Potato",
    "SaltShaker",
    "Spatula",
    "Spoon",
    "Tomato",
    "Pillow",
    "Book",
    "Laptop",
    "CreditCard",
    "KeyChain",
    "CellPhone",
    "RemoteControl",
    "ToiletPaper",
    "SoapBottle",
    "SoapBar",
    "HandTowel",
    "Towel",
    "Cloth",
    "Pen",
    "Pencil",
    "CD",
    "AlarmClock",
    "Newspaper",
    "Watch",
    "TissueBox",
    "Candle",
    "Vase",
    "Statue",
    "DishSponge",
    "SprayBottle",
    "ScrubBrush",
    "Plunger",
    "BaseballBat",
    "BasketBall",
    "TennisRacket",
    "TeddyBear",
    "Boots",
    "Bottle",
    "Kettle",
    "PaperTowelRoll",
    "WateringCan",
    "Ladle",
    "WineBottle",
    "AluminumFoil",
    "GarbageBag",
    "Dumbbell",
}

# Background/fixed fixtures.
# These can be structural, but should not become anchors/surfaces/containers by default.
BACKGROUND_FIXTURE_TYPES = {
    "Window",
    "Blinds",
    "Curtains",
    "LightSwitch",
    "Mirror",
    "Painting",
    "Poster",
    "Faucet",
    "ShowerHead",
    "ShowerDoor",
    "ShowerCurtain",
    "ShowerGlass",
}

# Candidate types by functional role.
ANCHOR_SEMANTIC_CANDIDATES = {
    "CounterTop",
    "Sink",
    "SinkBasin",
    "StoveBurner",
    "DiningTable",
    "Cabinet",
    "Drawer",
    "Fridge",
    "Microwave",
    "Shelf",
    "ShelvingUnit",
    "SideTable",
    "Desk",
    "Safe",

    "CoffeeTable",
    "TVStand",
    "Sofa",
    "ArmChair",
    "Chair",

    "Bed",
    "Dresser",

    "Toilet",
    "Bathtub",
    "BathtubBasin",
    "TowelHolder",
    "HandTowelHolder",
    "ToiletPaperHanger",

    "Box",
    "GarbageCan",
    "LaundryHamper",
}

STRUCTURAL_SEMANTIC_CANDIDATES = {
    "Cabinet",
    "CounterTop",
    "Drawer",
    "Fridge",
    "Microwave",
    "StoveBurner",
    "StoveKnob",
    "Sink",
    "SinkBasin",
    "Faucet",
    "CoffeeMachine",
    "Toaster",

    "Shelf",
    "ShelvingUnit",
    "SideTable",
    "DiningTable",
    "CoffeeTable",
    "Desk",
    "Dresser",
    "TVStand",
    "Safe",

    "Chair",
    "ArmChair",
    "Sofa",
    "Bed",
    "Stool",
    "Ottoman",
    "Footstool",

    "Toilet",
    "Bathtub",
    "BathtubBasin",
    "ShowerHead",
    "ShowerDoor",
    "ShowerCurtain",
    "ShowerGlass",
    "TowelHolder",
    "HandTowelHolder",
    "ToiletPaperHanger",

    "Window",
    "Blinds",
    "Curtains",
    "LightSwitch",
    "Mirror",
    "Painting",
    "Poster",

    "Television",
    "FloorLamp",
    "DeskLamp",
    "HousePlant",
    "RoomDecor",
    "Desktop",
    "VacuumCleaner",
    "GarbageCan",
    "LaundryHamper",
    "DogBed",
}

SURFACE_SEMANTIC_CANDIDATES = {
    "CounterTop",
    "Sink",
    "SinkBasin",
    "StoveBurner",

    "Shelf",
    "ShelvingUnit",
    "Desk",
    "DiningTable",
    "CoffeeTable",
    "SideTable",
    "TVStand",
    "Dresser",

    "Bed",
    "Sofa",
    "ArmChair",
    "Chair",
    "Stool",
    "Ottoman",
    "Footstool",

    "Bathtub",
    "BathtubBasin",
    "Toilet",
    "TowelHolder",
    "HandTowelHolder",
    "ToiletPaperHanger",
}

CONTAINER_SEMANTIC_CANDIDATES = {
    "Cabinet",
    "Drawer",
    "Fridge",
    "Microwave",
    "Safe",
    "Box",
    "GarbageCan",
    "LaundryHamper",

    "Cup",
    "Mug",
    "Bowl",
    "Pan",
    "Pot",

    "Sink",
    "SinkBasin",
    "Bathtub",
    "BathtubBasin",
}

# Types that are semantically possible but intentionally excluded from container-like
# because they may produce unnatural or undesired "in" relations.
CONTAINER_EXCLUDED_BY_LANGUAGE = {
    "Plate",
    "Toilet",
    "CoffeeMachine",
    "Toaster",
}

# 5.5 Sanity check: all semantic candidates must be observed

ALL_OBSERVED_TYPES = set(summary["global_object_type_counts"].keys())

ALL_CANDIDATE_TYPES = (
    ANCHOR_SEMANTIC_CANDIDATES
    | STRUCTURAL_SEMANTIC_CANDIDATES
    | SURFACE_SEMANTIC_CANDIDATES
    | CONTAINER_SEMANTIC_CANDIDATES
)

unobserved_candidates = sorted(ALL_CANDIDATE_TYPES - ALL_OBSERVED_TYPES)

print("Number of observed objectTypes:", len(ALL_OBSERVED_TYPES))
print("Number of candidate objectTypes before filtering:", len(ALL_CANDIDATE_TYPES))
print("Unobserved candidate types:", unobserved_candidates)

if unobserved_candidates:
    print("\nWARNING: These candidate types were not observed and will be ignored:")
    for t in unobserved_candidates:
        print(" -", t)

ANCHOR_SEMANTIC_CANDIDATES = ANCHOR_SEMANTIC_CANDIDATES & ALL_OBSERVED_TYPES
STRUCTURAL_SEMANTIC_CANDIDATES = STRUCTURAL_SEMANTIC_CANDIDATES & ALL_OBSERVED_TYPES
SURFACE_SEMANTIC_CANDIDATES = SURFACE_SEMANTIC_CANDIDATES & ALL_OBSERVED_TYPES
CONTAINER_SEMANTIC_CANDIDATES = CONTAINER_SEMANTIC_CANDIDATES & ALL_OBSERVED_TYPES

ALL_CANDIDATE_TYPES_AFTER_FILTERING = (
    ANCHOR_SEMANTIC_CANDIDATES
    | STRUCTURAL_SEMANTIC_CANDIDATES
    | SURFACE_SEMANTIC_CANDIDATES
    | CONTAINER_SEMANTIC_CANDIDATES
)

print("Number of candidate objectTypes after filtering:", len(ALL_CANDIDATE_TYPES_AFTER_FILTERING))

Number of observed objectTypes: 117
Number of candidate objectTypes before filtering: 61
Unobserved candidate types: []
Number of candidate objectTypes after filtering: 61


In [ ]:
# Selection function

def row_by_type(rows):
    return {row["object_type"]: row for row in rows}


row_map = row_by_type(rows)


def has_strong_support(object_type):
    row = row_map.get(object_type)
    return row is not None and row["support_level"] == "strong"


def has_medium_support(object_type):
    row = row_map.get(object_type)
    return row is not None and row["support_level"] == "medium"


def has_strong_or_medium_support(object_type):
    return has_strong_support(object_type) or has_medium_support(object_type)


def select_core(candidates):
    selected = set()

    for object_type in candidates:
        if object_type not in row_map:
            continue
        if object_type in GLOBAL_EXCLUDED_TYPES:
            continue
        if object_type in RELATION_PARTICIPANT_TYPES:
            continue
        if has_strong_support(object_type):
            selected.add(object_type)

    return selected


def select_optional(candidates):
    selected = set()

    for object_type in candidates:
        if object_type not in row_map:
            continue
        if object_type in GLOBAL_EXCLUDED_TYPES:
            continue
        if object_type in RELATION_PARTICIPANT_TYPES:
            continue
        if has_medium_support(object_type):
            selected.add(object_type)

    return selected


anchor_core = select_core(ANCHOR_SEMANTIC_CANDIDATES)
anchor_optional = select_optional(ANCHOR_SEMANTIC_CANDIDATES)

structural_core = select_core(STRUCTURAL_SEMANTIC_CANDIDATES)
structural_optional = select_optional(STRUCTURAL_SEMANTIC_CANDIDATES)

surface_core = select_core(SURFACE_SEMANTIC_CANDIDATES)
surface_optional = select_optional(SURFACE_SEMANTIC_CANDIDATES)

container_candidates_filtered = (
    CONTAINER_SEMANTIC_CANDIDATES
    - CONTAINER_EXCLUDED_BY_LANGUAGE
)

container_core = select_core(container_candidates_filtered)
container_optional = select_optional(container_candidates_filtered)

In [ ]:
def print_set(name, values):
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("{")
    for item in sorted(values):
        row = row_map[item]
        print(
            f'    "{item}",  '
            f'# scenes={row["scene_count"]}, '
            f'global={row["global_coverage"]:.1%}, '
            f'max_room={row["max_room"]}:{row["max_room_coverage"]:.1%}, '
            f'support={row["support_level"]}'
        )
    print("}")


print_set("ANCHOR_TYPES_CORE", anchor_core)
print_set("ANCHOR_TYPES_OPTIONAL", anchor_optional)

print_set("STRUCTURAL_TYPES_CORE", structural_core)
print_set("STRUCTURAL_TYPES_OPTIONAL", structural_optional)

print_set("SURFACE_TYPES_CORE", surface_core)
print_set("SURFACE_TYPES_OPTIONAL", surface_optional)

print_set("CONTAINER_LIKE_TYPES_CORE", container_core)
print_set("CONTAINER_LIKE_TYPES_OPTIONAL", container_optional)


ANCHOR_TYPES_CORE
{
    "ArmChair",  # scenes=32, global=27.8%, max_room=living_room:89.7%, support=strong
    "Bathtub",  # scenes=21, global=18.3%, max_room=bathroom:70.0%, support=strong
    "BathtubBasin",  # scenes=19, global=16.5%, max_room=bathroom:63.3%, support=strong
    "Bed",  # scenes=27, global=23.5%, max_room=bedroom:100.0%, support=strong
    "Box",  # scenes=41, global=35.7%, max_room=living_room:100.0%, support=strong
    "Cabinet",  # scenes=55, global=47.8%, max_room=kitchen:100.0%, support=strong
    "Chair",  # scenes=50, global=43.5%, max_room=bedroom:74.1%, support=strong
    "CoffeeTable",  # scenes=23, global=20.0%, max_room=living_room:79.3%, support=strong
    "CounterTop",  # scenes=55, global=47.8%, max_room=kitchen:100.0%, support=strong
    "Desk",  # scenes=23, global=20.0%, max_room=bedroom:77.8%, support=strong
    "DiningTable",  # scenes=28, global=24.3%, max_room=kitchen:51.7%, support=strong
    "Drawer",  # scenes=91, global=79.1%, max_room=bedr

In [ ]:
import csv
import os

out_dir = os.path.dirname(SUMMARY_PATH)
csv_path = os.path.join(out_dir, "_object_type_support_report.csv")

fieldnames = [
    "object_type",
    "instance_count",
    "scene_count",
    "global_coverage",
    "kitchen_scene_count",
    "living_room_scene_count",
    "bedroom_scene_count",
    "bathroom_scene_count",
    "kitchen_coverage",
    "living_room_coverage",
    "bedroom_coverage",
    "bathroom_coverage",
    "max_room",
    "max_room_coverage",
    "support_level",
    "pickupable_majority",
    "receptacle_majority",
    "moveable_majority",
    "openable_majority",
]

with open(csv_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for row in sorted(rows, key=lambda x: (-x["scene_count"], x["object_type"])):
        writer.writerow({key: row.get(key) for key in fieldnames})

print("\nWrote support report:", csv_path)


Wrote support report: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/object_type_inspection_20260602_164308/_object_type_support_report.csv


In [ ]:
# Generate config-ready object-type sets by support level.
#
# First run: use only strong types as the conservative core set.
# Later check: strong + medium, if broader coverage is needed.
# Weak types are kept out of the main config.

SUPPORT_LEVELS_TO_INCLUDE = {"strong"}

print("Support levels included:", SUPPORT_LEVELS_TO_INCLUDE)

# Helper functions

def get_supported_types(candidate_types, support_levels):
    selected = set()

    for object_type in candidate_types:
        row = row_map.get(object_type)

        if row is None:
            continue

        if row["support_level"] in support_levels:
            selected.add(object_type)

    return selected


def print_config_block(config_name, values):
    print("\n" + "#" * 80)
    print(f"# Paste into config.py: {config_name}")
    print("#" * 80)
    print(f"{config_name} = {{")
    for item in sorted(values):
        row = row_map[item]
        print(
            f'    "{item}",  '
            f'# support={row["support_level"]}, '
            f'scenes={row["scene_count"]}, '
            f'global={row["global_coverage"]:.1%}, '
            f'max_room={row["max_room"]}:{row["max_room_coverage"]:.1%}'
        )
    print("}")


container_candidates_filtered = (
    CONTAINER_SEMANTIC_CANDIDATES
    - CONTAINER_EXCLUDED_BY_LANGUAGE
)

ANCHOR_TYPES_SELECTED = get_supported_types(
    ANCHOR_SEMANTIC_CANDIDATES,
    SUPPORT_LEVELS_TO_INCLUDE,
)

STRUCTURAL_TYPES_SELECTED = get_supported_types(
    STRUCTURAL_SEMANTIC_CANDIDATES,
    SUPPORT_LEVELS_TO_INCLUDE,
)

SURFACE_TYPES_SELECTED = get_supported_types(
    SURFACE_SEMANTIC_CANDIDATES,
    SUPPORT_LEVELS_TO_INCLUDE,
)

CONTAINER_LIKE_TYPES_SELECTED = get_supported_types(
    container_candidates_filtered,
    SUPPORT_LEVELS_TO_INCLUDE,
)


print_config_block("ANCHOR_TYPES", ANCHOR_TYPES_SELECTED)
print_config_block("STRUCTURAL_TYPES", STRUCTURAL_TYPES_SELECTED)
print_config_block("SURFACE_TYPES", SURFACE_TYPES_SELECTED)
print_config_block("CONTAINER_LIKE_TYPES", CONTAINER_LIKE_TYPES_SELECTED)

Support levels included: {'strong'}

################################################################################
# Paste into config.py: ANCHOR_TYPES
################################################################################
ANCHOR_TYPES = {
    "ArmChair",  # support=strong, scenes=32, global=27.8%, max_room=living_room:89.7%
    "Bathtub",  # support=strong, scenes=21, global=18.3%, max_room=bathroom:70.0%
    "BathtubBasin",  # support=strong, scenes=19, global=16.5%, max_room=bathroom:63.3%
    "Bed",  # support=strong, scenes=27, global=23.5%, max_room=bedroom:100.0%
    "Box",  # support=strong, scenes=41, global=35.7%, max_room=living_room:100.0%
    "Cabinet",  # support=strong, scenes=55, global=47.8%, max_room=kitchen:100.0%
    "Chair",  # support=strong, scenes=50, global=43.5%, max_room=bedroom:74.1%
    "CoffeeTable",  # support=strong, scenes=23, global=20.0%, max_room=living_room:79.3%
    "CounterTop",  # support=strong, scenes=55, global=47.8%, max_room=kit

In [ ]:
import os
import shutil
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive"
assert os.path.exists(DRIVE_ROOT), (
    "Google Drive is not mounted. "
    "Please run drive.mount('/content/drive') first."
)

print("CSV path:", csv_path)
print("CSV exists:", os.path.exists(csv_path))
assert os.path.exists(csv_path), f"CSV report not found: {csv_path}"

print("Summary path:", SUMMARY_PATH)
print("Summary exists:", os.path.exists(SUMMARY_PATH))

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

drive_out_dir = os.path.join(
    DRIVE_ROOT,
    "Colab Notebooks",
    "linear_probe_full",
    "data",
    f"object_type_selection_report_{timestamp}",
)

os.makedirs(drive_out_dir, exist_ok=True)

dst_csv_path = os.path.join(drive_out_dir, os.path.basename(csv_path))
shutil.copy2(csv_path, dst_csv_path)

print("Copied CSV report to:")
print(dst_csv_path)

if os.path.exists(SUMMARY_PATH):
    dst_summary_path = os.path.join(drive_out_dir, os.path.basename(SUMMARY_PATH))
    shutil.copy2(SUMMARY_PATH, dst_summary_path)

    print("Copied summary file to:")
    print(dst_summary_path)

print("\nDone.")
print("Google Drive output directory:")
print(drive_out_dir)

CSV path: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/object_type_inspection_20260602_164308/_object_type_support_report.csv
CSV exists: True
Summary path: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data_prepare_step1_3/data/object_type_inspection_20260602_164308/_object_type_summary.json
Summary exists: True
Copied CSV report to:
/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/object_type_selection_report_20260602_192318/_object_type_support_report.csv
Copied summary file to:
/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/object_type_selection_report_20260602_192318/_object_type_summary.json

Done.
Google Drive output directory:
/content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/object_type_selection_report_20260602_192318


In [ ]:
import os
import shutil

zip_base = drive_out_dir
zip_path = shutil.make_archive(
    base_name=zip_base,
    format="zip",
    root_dir=drive_out_dir,
)

print("ZIP created:", zip_path)
print("Size MB:", os.path.getsize(zip_path) / 1024 / 1024)

from google.colab import files
files.download(zip_path)

ZIP created: /content/drive/MyDrive/Colab Notebooks/linear_probe_full/data/object_type_selection_report_20260602_192318.zip
Size MB: 0.1215677261352539


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>